# The branch order inside one object rule

The earlier branch admits an admin; the later branch admits an EU manager. The order decides which caller is let through.

[Script](17_branch_order.py)


## Imports


In [ ]:
"""The branch order inside one object rule."""

from __future__ import annotations

import asyncio

from pydantic import Field

from aoa.action_machine.auth import ApplicationRole
from aoa.action_machine.context import Context
from aoa.action_machine.context.context_view import ContextView
from aoa.action_machine.context.user_info import UserInfo
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.access_control import Allowed, Refused, Verdict
from aoa.action_machine.intents.access_decide import access_decide
from aoa.action_machine.intents.aspects import summary_aspect
from aoa.action_machine.intents.check_roles import check_roles
from aoa.action_machine.intents.context_requires import context_requires
from aoa.action_machine.intents.meta import meta
from aoa.action_machine.model import BaseAction, BaseParams, BaseResult
from aoa.action_machine.model.base_state import BaseState
from aoa.action_machine.resources import Connections
from aoa.action_machine.runtime.action_product_machine import ActionProductMachine
from aoa.action_machine.runtime.tools_box import ToolsBox


## The rule

The earlier branch admits an admin; the later branch admits an EU manager. The order decides which caller is let through.


In [ ]:


class StoreDomain(BaseDomain):
    """Group the order operations."""

    name = "store"
    description = "Order management"


class ManagerRole(ApplicationRole):
    """Permit order management."""

    name = "manager"
    description = "Can manage orders"


class AdminRole(ManagerRole):
    """Include manager permissions and bypass the team restriction."""

    name = "admin"
    description = "Global administrator"


class OrderParams(BaseParams):
    """Identify the order to cancel."""

    order_id: str = Field(description="Order identifier")


class OrderResult(BaseResult):
    """Report the cancelled order."""

    order_id: str = Field(description="Cancelled order identifier")


@meta(description="Cancel an order", domain=StoreDomain)
@check_roles(ManagerRole)
class CancelOrderAction(BaseAction[OrderParams, OrderResult]):
    """Cancel an order after checking access."""

    @access_decide("An admin is always admitted; a manager must be in the EU team")
    @context_requires("user.user_id", "user.roles")
    async def cancel_order_access_decide(
        self,
        params: OrderParams,
        box: ToolsBox,
        connections: Connections,
        ctx: ContextView,
    ) -> Verdict:
        """The earlier branch admits an admin; the later one admits an EU manager."""
        if any(role.__name__ == "AdminRole" for role in ctx.get("user.roles")):
            return Allowed()
        if ctx.get("user.user_id").startswith("eu-"):
            return Allowed()
        return Refused("EU_TEAM_ONLY")

    @summary_aspect("Cancel the order")
    async def cancel_summary(self, params: OrderParams, state: BaseState, box: ToolsBox, connections: Connections) -> OrderResult:
        """Return the cancellation result."""
        return OrderResult(order_id=params.order_id)


## Try it

Run the cells in order. The calls below are the same as in the script.


In [ ]:
machine = ActionProductMachine(cache_coordinator=None)
manager = Context(user=UserInfo(user_id="us-m1", roles=(ManagerRole,)))
admin = Context(user=UserInfo(user_id="us-a1", roles=(AdminRole,)))
params = OrderParams(order_id="ord-001")

answer = await machine.check_access_decide(manager, CancelOrderAction, params)
print("US manager:", answer.model_dump(mode="json"))
answer = await machine.check_access_decide(admin, CancelOrderAction, params)
print("US admin:", answer.model_dump(mode="json"))


US manager: {'kind': 'refused', 'gate': 'ACCESS_DECIDE', 'reason': 'EU_TEAM_ONLY'}
US admin: {'kind': 'allowed'}
